# 00 — Resume the reduced inverse study

This revision starts from **your uploaded `heat.zip`**. It preserves ten completed production trainings (six smooth 2D and four nonlinear 1D), their checkpoints and evaluations, twelve completed linear classical fits, and your verified 1601-node nonlinear reference.

The declared core study has **39 neural trainings in total**, including the ten already completed: 18 smooth, 12 nonlinear, and 9 brass. On the supplied archive, **29 trainings remain**. The live inventory below recalculates these counts on every visit. The composite case remains excluded.

## Run order

| Notebook | Action in this revision |
|---|---|
| 00 | Check sources, preserved results, and the reduced plan; no training |
| 01 | Reuse and verify the existing forward reference; no new reference generation |
| 02 | Reuse the eight linear classical fits needed by the reduced study |
| 03 | Run only the 12 missing smooth trainings; independently replay all selected checkpoints |
| 04 | Run only the 8 missing nonlinear trainings; four per-dataset classical comparisons; cached replay refinement |
| 05 | Run 9 brass trainings; evaluate and replay every completed model |
| 06 | Optional four dimensional demonstrations; training stays OFF |
| 07 | Export coverage, every result, descriptive spread, and paired differences |

**Notebooks 03–05 are production-ready:** their main training switches are already `True`. Run them one at a time on one GPU. Re-running a notebook skips matching completed jobs and resumes matching interrupted jobs. `MAX_NEW_JOBS` (or `MAX_NEW_TRAINING_JOBS` in 05) limits pending jobs after completed jobs have been removed from the queue; `1` therefore means the next pending job.

Use your existing working Python environment. All six original solver/data modules remain unchanged. Do not change training budgets, data, or solver files halfway through this cohort: the resume guard will report a conflict rather than silently create a replacement experiment. Copying the entire extracted folder to a different machine is supported.

The three repetitions in a noisy case use three predetermined `(noise_seed, training_seed)` pairs. They assess combined variability. The report uses individual values and descriptive summaries; it does not treat this design as a full crossed random-effects experiment or promise precise confidence intervals.


In [ ]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "inverse_study").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Extract the entire study ZIP and open this notebook from its folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from inverse_study.common import *
from inverse_study import numerics as num
from inverse_study import workflows as flow
RESULTS.mkdir(exist_ok=True)
plt.rcParams.update({"font.size": 11, "figure.dpi": 110, "axes.grid": True,
                     "grid.alpha": 0.2, "savefig.bbox": "tight"})
print("Study folder:", ROOT)
from inverse_study import completion as study
from inverse_study import completion_checks as checks
REDUCED = RESULTS / "reduced_plan"
REDUCED.mkdir(parents=True, exist_ok=True)
study.verify_protocol_lock(require_reference=True)


In [ ]:
versions = environment()
display(pd.Series(versions["packages"], name="installed_version").to_frame())
write_json(REDUCED / "environment.json", versions)
print("Missing TensorFlow is acceptable for notebooks 00–02 and the numerical parts of 04–07.")

## 1. Verify the bundled sources

Fifteen archived non-composite PINN runs are included. They remain **legacy evidence**, because the original noisy terminal arrays include boundary samples. The new protocol uses only interior noisy observations and a separate known boundary history for every method. The shared 5%/33×33 run is one run, even when shown in both the noise and observation-density figures.

In [ ]:
source_manifest = read_json(DATA / "SOURCE_MANIFEST.json")
checks = []
for item in source_manifest:
    if "destination" not in item:
        continue
    path = ROOT / item["destination"]
    checks.append({"file": item["destination"], "hash_ok": sha256_file(path) == item["sha256"]})
assert all(r["hash_ok"] for r in checks), "A bundled source file changed. Inspect before proceeding."
inventory = archived_inventory()
display(inventory[["run_id", "n_all", "n_interior", "boundary_noise_norm", "original_initial_relative_l2"]])
inventory.to_csv(REDUCED / "legacy_inventory.csv", index=False)
print(len(checks), "bundled source files checked;", len(inventory), "archived runs.")

## 2. The common observation contract

For a full grid with `n` points per axis, only `(n−2)^d` interior samples enter the terminal data loss. Boundary histories enter their own loss or boundary constraints. The same stored array is passed to the neural and classical methods.

Synthetic noise is iid Gaussian with $\sigma=\rho\,\mathrm{RMS}(y_\mathrm{clean})$. The actual known synthetic noise norm $\delta=\|\epsilon\|_2$ is saved separately from the nominal level $\rho$. The discrepancy selector uses this norm; it never uses the true initial-field error. Clean data use a stated numerical floor. Brass requires an independently justified uncertainty estimate, not this synthetic rule.

`observations.npz` contains only observed values, positions and noise metadata. `evaluation_only.npz` holds the clean terminal vector/noise for post-fit checks. Do not use the latter or any initial truth for checkpoint or hyperparameter selection.

In [ ]:
problem = problem_spec("smooth2d")
dataset_folder = freeze_observations(problem, obs_per_dim=33, noise_level=.05, noise_seed=3100)
_, observations, meta = load_observations(dataset_folder)
assert len(observations["y_obs"]) == 31**2
assert np.all((observations["X_obs"] > 0) & (observations["X_obs"] < 1))
from inverse_study.pinn import validate_inputs
display(pd.Series(meta).to_frame("value"))
print(validate_inputs(problem, observations, {"profile": "smoke"}))
plt.figure(figsize=(6, 5))
plt.scatter(*observations["X_obs"].T, c=observations["y_obs"], s=13, cmap="viridis")
plt.colorbar(label="Observed terminal temperature")
plt.xlabel("x"); plt.ylabel("y"); plt.title("One frozen interior observation vector")
plt.show()

## 3. Frozen reduced plan and live remaining queue

Clean synthetic observations use training seeds 2026, 2027 and 2028. Noisy synthetic observations use the three pairs `(3100, 2026)`, `(3101, 2027)` and `(3102, 2028)`, shared across methods. All original completed screening jobs belong to this cohort.

The plan was reduced by seed identifiers, independently of which method has a smaller initial error. Report every selected outcome and every failed job. Extra archived classical noise draws remain on disk and are excluded from the reduced primary cohort. Brass has three training seeds on **one physical experiment**, not three experimental replications.

A completed flag alone is insufficient for reuse. The queue checks source identity, full configuration, observations, evaluation grid, completed budget, and saved artifacts. Conflicts stop execution for inspection. Missing post-training evaluation can be repaired without retraining a completed model.


In [ ]:
linear_plan = study.compact_linear_plan()
nonlinear_plan = study.compact_nonlinear_plan()
core_plan = study.compact_core_plan()
for label, table in [("linear",linear_plan),("nonlinear",nonlinear_plan),("synthetic_core",core_plan)]:
    table.to_csv(REDUCED / f"{label}_plan.csv",index=False)
inventory = study.plan_inventory(core_plan)
inventory.to_csv(REDUCED / "synthetic_inventory.csv",index=False)
from inverse_study.completion_report import brass_cohort
brass_inventory, _, _, _ = brass_cohort()
brass_inventory.to_csv(REDUCED / "brass_inventory.csv",index=False)
summary = []
for label, frame in [("smooth2d",inventory[inventory.case.eq("smooth2d")]),
                      ("nonlinear1d",inventory[inventory.case.eq("nonlinear1d")]),
                      ("brass",brass_inventory)]:
    summary.append({"case":label,"planned":len(frame),
                    "completed":int(frame.status.eq("completed").sum()),
                    "missing_or_resumable":int(frame.status.isin(["missing","resumable"]).sum()),
                    "blocked":int(frame.status.eq("blocked").sum())})
summary = pd.DataFrame(summary)
display(summary)
print("Core trainings planned:",int(summary.planned.sum()),
      "| completed:",int(summary.completed.sum()),
      "| still pending:",int(summary.missing_or_resumable.sum()))
display(inventory[["case","method","noise_level","noise_seed","training_seed","status","reason"]])
print("This notebook does not start any training.")


## Protocol changes that must be reported in the final paper

The prospective comparison corrects noisy-boundary handling; separates noise, initialization, Fourier-feature and collocation seeds; uses scrambled Sobol points paired across methods; and applies the same **unit-time PDE residual**, `T * (theta_t − Fo div(a grad theta))`, to every neural control. This is an explicit loss scaling, distinct from the archived residual convention. Final fixed-budget checkpoints are primary. A separate truth-free best checkpoint is diagnostic.

The new independent nonlinear forward solver uses conservative Kirchhoff fluxes and BDF. The rich sine-basis baselines and nonlinear finite-parameter comparator are new implementations. Neither their presence nor a passing software check certifies the old FreeFEM QR results. Report these distinctions when replacing the manuscript's provisional methods/results.